In [39]:
import tensorflow as tf
import os
from hyperopt import STATUS_OK,Trials,fmin,hp,tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from mlflow.models import infer_signature
import pandas as pd
import numpy as np
import mlflow

In [40]:
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

In [41]:
print("GPU Available: ", tf.config.list_physical_devices('GPU'))

GPU Available:  []


In [42]:
data=pd.read_csv("https://raw.githubusercontent.com/Arannamoy/datasets/refs/heads/main/white_wine_quality.csv",sep=";")

In [43]:
data.head(5)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [44]:
# split the data into training,validation, test sets

train,test=train_test_split(data,test_size=0.3,random_state=42)

In [45]:
train_x=train.drop(columns=['quality'],axis=1).values
train_y=train[['quality']].values.ravel()
test_x=test.drop(columns=['quality'],axis=1).values
test_y=test[['quality']].values.ravel()

train_x,valid_x,train_y,valid_y=train_test_split(train_x,train_y,test_size=0.3,random_state=42)
signature=infer_signature(train_x,train_y)

In [46]:
def train_model(params,epochs,train_x,train_y,valid_x,valid_y,test_x,test_y):

    # define model architecture
    mean=np.mean(train_x,axis=0)
    var=np.var(train_x,axis=0)
    model=tf.keras.Sequential(
        [
            tf.keras.Input([train_x.shape[1]]),
            tf.keras.layers.Normalization(mean=mean,variance=var),
            tf.keras.layers.Dense(64,activation='relu'),
            tf.keras.layers.Dense(1)
        ]
    )


    model.compile(optimizer=tf.keras.optimizers.SGD
    (
        learning_rate=params["lr"],momentum=params["momentum"]
    ),
    loss="mean_squared_error",
    metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    with mlflow.start_run(nested=True):
        model.fit(train_x,train_y,validation_data=(valid_x,valid_y),epochs=epochs,batch_size=64)

        eval_result=model.evaluate(valid_x,valid_y,batch_size=64)
        eval_rmse=eval_result[1]


        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse",eval_rmse)
        model_info=mlflow.tensorflow.log_model(
            model,
            artifact_path="002 White Wine ANN",
            signature=signature
        )


        return {"loss":eval_rmse,"status":STATUS_OK,"model":model}

In [47]:
def objective(params):
    result=train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y
    )
    return result

In [48]:
space={
    "lr":hp.loguniform("lr",np.log(1e-5),np.log(1e-1)),
    "momentum":hp.uniform("momentum",0.0,1.0)
}

In [49]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(experiment_name="002 White Wine Quality Using ANN")

<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1774278601595, experiment_id='5', last_update_time=1774278601595, lifecycle_stage='active', name='002 White Wine Quality Using ANN', tags={}, workspace='default'>

In [ ]:
with mlflow.start_run():
    trials=Trials()
    best=fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )


    best_run=sorted(trials.results,key=lambda x:x["loss"])[0]

    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse",best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"],"model",signature=signature)



    print(f"Best params: {best}")
    print(f"Best eval rmse: {best_run["loss"]}")

Epoch 1/3                                            

 1/38 ━━━━━━━━━━━━━━━━━━━━ 8s 226ms/step - loss: 34.7410 - root_mean_squared_error: 5.8941
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 20.5891 - root_mean_squared_error: 4.5375 - val_loss: 4.4442 - val_root_mean_squared_error: 2.1081

Epoch 2/3                                            

 1/38 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 3.9789 - root_mean_squared_error: 1.9947
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.1278 - root_mean_squared_error: 2.0317 - val_loss: 4.2336 - val_root_mean_squared_error: 2.0576

Epoch 3/3                                            

 1/38 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 5.0709 - root_mean_squared_error: 2.2519
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.4093 - root_mean_squared_error: 1.5522 - val_loss: 1.8498 - val_root_mean_squared_error: 1.3601

 1/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 2.4872 - root_mean_squared_error: 1.5771
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

2026/03/23 21:18:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run charming-bear-666 at: http://127.0.0.1:5000/#/experiments/5/runs/4083e3b448544013b5b4ce15628d3104

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5

Epoch 1/3                                                                      

 1/38 ━━━━━━━━━━━━━━━━━━━━ 8s 218ms/step - loss: 38.6702 - root_mean_squared_error: 6.2185
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.6238 - root_mean_squared_error: 2.3715 - val_loss: 1.4737 - val_root_mean_squared_error: 1.2139

Epoch 2/3                                                                      

 1/38 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 1.1444 - root_mean_squared_error: 1.0698
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.2480 - root_mean_squared_error: 1.1171 - val_loss: 1.1147 - val_root_mean_squared_error: 1.0558

Epoch 3/3                                                                      

 1/38 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 1.2045 - root_mean_squared_error: 1.0975
38/38 ━━━━━━━━━━━━━━━━━━━

2026/03/23 21:18:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run rebellious-ant-935 at: http://127.0.0.1:5000/#/experiments/5/runs/2033174c561a447c9696c854a307ab9a

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5                   

Epoch 1/3                                                                      

 1/38 ━━━━━━━━━━━━━━━━━━━━ 8s 241ms/step - loss: 34.4226 - root_mean_squared_error: 5.8671
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 11.6678 - root_mean_squared_error: 3.4158 - val_loss: 2.5842 - val_root_mean_squared_error: 1.6075

Epoch 2/3                                                                      

 1/38 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 3.5030 - root_mean_squared_error: 1.8716
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.1716 - root_mean_squared_error: 1.4736 - val_loss: 1.8265 - val_root_mean_squared_error: 1.3515

Epoch 3/3                                                                      

 1/38 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 1.1091 - root_mean_squared_error: 1.0531
38/3

...

2026/03/23 21:18:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run receptive-worm-584 at: http://127.0.0.1:5000/#/experiments/5/runs/43749dca4ccc46f5ae76e2561f3b26fc

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5                   

100%|██████████| 4/4 [00:22<00:00,  5.69s/trial, best loss: 0.9622619152069092]

2026/03/23 21:18:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best params: {'lr': np.float64(0.010997402896875811), 'momentum': np.float64(0.2590494615227876)}
Best eval rmse: 0.9622619152069092
🏃 View run awesome-grub-696 at: http://127.0.0.1:5000/#/experiments/5/runs/0db439acab2948d089ad644b1c180a9c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


: 